In [0]:
from pyspark.sql import functions as F

dim_customer = (spark.read.table("northstar.silver.customers")
    .select("customer_id", "customer_unique_id",
            "customer_city", "customer_state", "customer_zip_code_prefix"))

dim_customer.write.mode("overwrite").saveAsTable("northstar.gold.dim_customer")
print("✅ gold.dim_customer")

In [0]:
from pyspark.sql import functions as F

orders  = spark.read.table("northstar.silver.orders")
reviews = spark.read.table("northstar.silver.reviews")

# One review score per order
review_by_order = reviews.groupBy("order_id").agg(F.avg("review_score").alias("review_score"))

fct = (orders
    .join(review_by_order, "order_id", "left")                       # JOIN reviews onto orders
    .withColumn("delivery_days",
        F.datediff("order_delivered_customer_date", "order_purchase_timestamp"))
    .withColumn("estimated_days",
        F.datediff("order_estimated_delivery_date", "order_purchase_timestamp"))
    .withColumn("is_delivered", (F.col("order_status") == "delivered").cast("int"))
    .withColumn("is_late",
        F.when(F.col("order_delivered_customer_date") > F.col("order_estimated_delivery_date"), 1)
         .otherwise(0))                                              # ← the ML label later!
    .select("order_id", "customer_id", "order_status",
            "order_purchase_timestamp", "order_delivered_customer_date",
            "order_estimated_delivery_date",
            "delivery_days", "estimated_days", "is_delivered", "is_late", "review_score"))

fct.write.mode("overwrite").saveAsTable("northstar.gold.fct_orders")
print("✅ gold.fct_orders")

In [0]:
from pyspark.sql import functions as F

fct  = spark.read.table("northstar.gold.fct_orders")
cust = spark.read.table("northstar.gold.dim_customer")

agg = (fct.groupBy("customer_id").agg(
        F.count("*").alias("total_orders"),
        F.sum("is_delivered").alias("delivered_orders"),
        F.sum("is_late").alias("late_orders"),
        F.round(F.avg("delivery_days"), 1).alias("avg_delivery_days"),
        F.round(F.avg("review_score"), 2).alias("avg_review_score")))

customer_360 = (cust.join(agg, "customer_id", "left")
    .withColumn("late_rate", F.round(F.col("late_orders") / F.col("total_orders"), 3))
    .fillna(0, ["total_orders", "delivered_orders", "late_orders"]))

customer_360.write.mode("overwrite").saveAsTable("northstar.gold.customer_360")
print("✅ gold.customer_360")

In [0]:
%sql
-- 1) Overall late-delivery rate
SELECT SUM(is_late) AS late, COUNT(*) AS total,
       ROUND(100.0 * SUM(is_late)/COUNT(*), 1) AS late_pct
FROM northstar.gold.fct_orders WHERE is_delivered = 1;

-- 2) Avg review score by state (fact JOIN dimension)
SELECT c.customer_state,
       ROUND(AVG(f.review_score), 2) AS avg_review,
       COUNT(*) AS orders
FROM northstar.gold.fct_orders f
JOIN northstar.gold.dim_customer c ON f.customer_id = c.customer_id
GROUP BY c.customer_state ORDER BY orders DESC LIMIT 10;

In [0]:
# 1) Monthly order trend
display(spark.sql("""
  SELECT date_trunc('month', order_purchase_timestamp) AS month, COUNT(*) AS orders
  FROM northstar.gold.fct_orders GROUP BY 1 ORDER BY 1
"""))

# 2) ⭐ Does LATE delivery hurt reviews?  (the insight that justifies the ML model)
display(spark.sql("""
  SELECT is_late, ROUND(AVG(review_score),2) AS avg_review, COUNT(*) AS orders
  FROM northstar.gold.fct_orders
  WHERE review_score IS NOT NULL AND is_delivered = 1
  GROUP BY is_late
"""))

# 3) Review score distribution (1–5)
display(spark.sql("""
  SELECT review_score, COUNT(*) AS n
  FROM northstar.silver.reviews GROUP BY review_score ORDER BY review_score
"""))

# 4) Best states by on-time delivery (lowest late %)
display(spark.sql("""
  SELECT c.customer_state, COUNT(*) AS delivered,
         ROUND(100.0*SUM(f.is_late)/COUNT(*),1) AS late_pct
  FROM northstar.gold.fct_orders f
  JOIN northstar.gold.dim_customer c ON f.customer_id = c.customer_id
  WHERE f.is_delivered = 1
  GROUP BY c.customer_state HAVING COUNT(*) > 100
  ORDER BY late_pct ASC LIMIT 10
"""))

# 5) Order status breakdown
display(spark.sql("""
  SELECT order_status, COUNT(*) AS orders
  FROM northstar.gold.fct_orders GROUP BY order_status ORDER BY orders DESC
"""))

# 6) 💰 Revenue by month (pulls price from order_items)
display(spark.sql("""
  SELECT date_trunc('month', o.order_purchase_timestamp) AS month,
         ROUND(SUM(try_cast(oi.price AS double)), 2) AS revenue
  FROM northstar.bronze.order_items_raw oi
  JOIN northstar.gold.fct_orders o ON oi.order_id = o.order_id
  GROUP BY 1 ORDER BY 1
"""))

In [0]:
from pyspark.sql import functions as F

# dim_product
(spark.read.table("northstar.silver.products")
  .select("product_id","category","weight_g","photos_qty")
  .write.mode("overwrite").saveAsTable("northstar.gold.dim_product"))

# dim_seller
(spark.read.table("northstar.silver.sellers")
  .select("seller_id","seller_city","seller_state")
  .write.mode("overwrite").saveAsTable("northstar.gold.dim_seller"))

# dim_date (generated from order dates — a classic dimension)
(spark.read.table("northstar.silver.orders")
  .select(F.to_date("order_purchase_timestamp").alias("date"))
  .filter("date IS NOT NULL").distinct()
  .withColumn("year",  F.year("date"))
  .withColumn("month", F.month("date"))
  .withColumn("day_of_week", F.date_format("date","EEEE"))
  .withColumn("month_name",  F.date_format("date","MMMM"))
  .write.mode("overwrite").saveAsTable("northstar.gold.dim_date"))
print("✅ Gold dims: dim_product, dim_seller, dim_date")

In [0]:
from pyspark.sql import functions as F

items  = spark.read.table("northstar.silver.order_items")
orders = spark.read.table("northstar.silver.orders")

(items
  .join(orders.select("order_id","customer_id","order_status","order_purchase_timestamp"),
        "order_id","left")
  .withColumn("revenue", F.col("price") + F.col("freight_value"))   # money per line item
  .withColumn("order_date", F.to_date("order_purchase_timestamp"))
  .select("order_id","order_item_id","product_id","seller_id","customer_id",
          "order_status","order_date","price","freight_value","revenue")
  .write.mode("overwrite").saveAsTable("northstar.gold.fct_order_items"))
print("✅ gold.fct_order_items")

In [0]:
from pyspark.sql import functions as F

fi = spark.read.table("northstar.gold.fct_order_items")

# Sales by product category
(fi.join(spark.read.table("northstar.gold.dim_product"), "product_id", "left")
  .groupBy("category").agg(
     F.round(F.sum("revenue"),2).alias("total_revenue"),
     F.count("*").alias("items_sold"),
     F.round(F.avg("price"),2).alias("avg_price"))
  .orderBy(F.desc("total_revenue"))
  .write.mode("overwrite").saveAsTable("northstar.gold.sales_by_category"))

# Daily revenue
(fi.groupBy("order_date").agg(
     F.round(F.sum("revenue"),2).alias("revenue"),
     F.countDistinct("order_id").alias("orders"))
  .orderBy("order_date")
  .write.mode("overwrite").saveAsTable("northstar.gold.daily_revenue"))

# Seller performance
(fi.join(spark.read.table("northstar.gold.dim_seller"), "seller_id", "left")
  .groupBy("seller_id","seller_state").agg(
     F.round(F.sum("revenue"),2).alias("total_revenue"),
     F.countDistinct("order_id").alias("orders"),
     F.count("*").alias("items_sold"))
  .orderBy(F.desc("total_revenue"))
  .write.mode("overwrite").saveAsTable("northstar.gold.seller_performance"))
print("✅ Gold aggregates: sales_by_category, daily_revenue, seller_performance")

In [0]:
%sql
SELECT * FROM northstar.gold.sales_by_category LIMIT 10;

In [0]:
%sql
SELECT * FROM northstar.gold.seller_performance ORDER BY total_revenue DESC LIMIT 10;